In [ ]:
!sudo apt-get update && sudo apt-get install -y libhunspell-dev


Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:2 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Hit:6 https://cli.github.com/packages stable InRelease
Get:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:10 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,297 kB]
Get:11 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [80.2 kB]
Get:12 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease [24.3 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 

In [ ]:
!pip install hunspell


  Preparing metadata (setup.py) ... done
  Created wheel for hunspell: filename=hunspell-0.5.5-cp312-cp312-linux_x86_64.whl size=66944 sha256=5d33c42a3227f15294a1b4099a6cf7dda5fb89b5898cf3829ecf5eb5120d3b52
  Stored in directory: /root/.cache/pip/wheels/a4/1e/1c/3438c8e5af66b88b3bab7d99b7c2ec24a9bfe669b57c3f87a6
Successfully built hunspell


In [ ]:
# Hunspell Spanish Spell Correction Setup with Accuracy Metrics
# Works on Windows, Kaggle, and Colab

import pandas as pd
import requests
import os
from pathlib import Path
import numpy as np
from collections import defaultdict


In [ ]:

# Step 1: Install required packages
try:
    import hunspell
except ImportError:
    print("Installing pyhunspell...")
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pyhunspell"])
    import hunspell


In [ ]:

# Step 2: Download Spanish dictionary files
def download_spanish_dictionary():
    """Download Spanish dictionary files from the GitHub repo"""

    # URLs for the Spanish dictionary files
    dic_url = "https://raw.githubusercontent.com/wooorm/dictionaries/main/dictionaries/es/index.dic"
    aff_url = "https://raw.githubusercontent.com/wooorm/dictionaries/main/dictionaries/es/index.aff"

    # Create dictionaries directory
    dict_dir = Path("dictionaries")
    dict_dir.mkdir(exist_ok=True)

    # Download .dic file
    print("Downloading Spanish dictionary (.dic)...")
    dic_response = requests.get(dic_url)
    dic_response.raise_for_status()

    with open(dict_dir / "es.dic", "wb") as f:
        f.write(dic_response.content)

    # Download .aff file
    print("Downloading Spanish affix file (.aff)...")
    aff_response = requests.get(aff_url)
    aff_response.raise_for_status()

    with open(dict_dir / "es.aff", "wb") as f:
        f.write(aff_response.content)

    print("Dictionary files downloaded successfully!")
    return str(dict_dir / "es")


In [ ]:

# Step 3: Initialize Hunspell with Spanish dictionary
def setup_spanish_hunspell():
    """Setup Hunspell with Spanish dictionary"""

    # Download dictionary if not exists
    if not os.path.exists("dictionaries/es.dic") or not os.path.exists("dictionaries/es.aff"):
        dict_path = download_spanish_dictionary()
    else:
        dict_path = "dictionaries/es"

    # Initialize Hunspell
    try:
        hobj = hunspell.HunSpell(dict_path + ".dic", dict_path + ".aff")
        print("Spanish Hunspell initialized successfully!")
        return hobj
    except Exception as e:
        print(f"Error initializing Hunspell: {e}")
        # Alternative: try system hunspell if available
        try:
            hobj = hunspell.HunSpell("es", "es")
            print("Using system Spanish dictionary")
            return hobj
        except:
            print("Could not initialize Hunspell. Please check installation.")
            return None

# Step 4: Spell correction functions
def correct_word(word, hunspell_obj):
    """Correct a single word using Hunspell"""
    if not hunspell_obj:
        return word

    # Check if word is correct
    if hunspell_obj.spell(word):
        return word

    # Get suggestions
    suggestions = hunspell_obj.suggest(word)

    # Return best suggestion or original word
    return suggestions[0] if suggestions else word

def correct_text(text, hunspell_obj):
    """Correct all words in a text string"""
    if not text or not hunspell_obj:
        return text

    words = text.split()
    corrected_words = []

    for word in words:
        # Keep punctuation
        clean_word = word.strip('.,!?;:"()[]{}')
        prefix = word[:len(word) - len(word.lstrip('.,!?;:"()[]{}'))]
        suffix = word[len(clean_word) + len(prefix):]

        if clean_word:
            corrected = correct_word(clean_word, hunspell_obj)
            corrected_words.append(prefix + corrected + suffix)
        else:
            corrected_words.append(word)

    return ' '.join(corrected_words)

# Step 5: Accuracy calculation functions
def calculate_word_accuracy(predicted, target):
    """Calculate word-level accuracy"""
    pred_words = predicted.strip().split()
    target_words = target.strip().split()

    if len(target_words) == 0:
        return 1.0 if len(pred_words) == 0 else 0.0

    # For different lengths, consider only common positions
    min_len = min(len(pred_words), len(target_words))
    max_len = max(len(pred_words), len(target_words))

    correct_words = sum(1 for i in range(min_len) if pred_words[i] == target_words[i])

    # Penalize for length differences
    accuracy = correct_words / max_len
    return accuracy

def calculate_character_accuracy(predicted, target):
    """Calculate character-level accuracy using edit distance"""
    if len(target) == 0:
        return 1.0 if len(predicted) == 0 else 0.0

    # Simple character-level accuracy
    min_len = min(len(predicted), len(target))
    max_len = max(len(predicted), len(target))

    if max_len == 0:
        return 1.0

    correct_chars = sum(1 for i in range(min_len) if predicted[i] == target[i])
    accuracy = correct_chars / max_len
    return accuracy

def calculate_exact_match_accuracy(predicted, target):
    """Calculate exact string match accuracy"""
    return 1.0 if predicted.strip() == target.strip() else 0.0

def calculate_metrics(df):
    """Calculate comprehensive accuracy metrics"""

    # Individual accuracies
    word_accuracies = []
    char_accuracies = []
    exact_matches = []

    for _, row in df.iterrows():
        pred = str(row['input_corrected'])
        target = str(row['target'])

        word_acc = calculate_word_accuracy(pred, target)
        char_acc = calculate_character_accuracy(pred, target)
        exact_match = calculate_exact_match_accuracy(pred, target)

        word_accuracies.append(word_acc)
        char_accuracies.append(char_acc)
        exact_matches.append(exact_match)

    # Store individual metrics
    df['word_accuracy'] = word_accuracies
    df['char_accuracy'] = char_accuracies
    df['exact_match'] = exact_matches

    # Calculate overall metrics
    metrics = {
        'total_samples': len(df),
        'exact_match_accuracy': np.mean(exact_matches),
        'word_level_accuracy': np.mean(word_accuracies),
        'character_level_accuracy': np.mean(char_accuracies),
        'perfect_corrections': sum(exact_matches),
        'perfect_correction_rate': sum(exact_matches) / len(df) * 100
    }

    return metrics


In [ ]:

# Step 6: Process your TSV file (FIXED VERSION)
def process_tsv_file(file_path, hunspell_obj, nrows=None):
    """Process TSV file and add spell-corrected input column with accuracy metrics"""

    # Read TSV file
    print(f"Reading TSV file: {file_path}")
    df = pd.read_csv(file_path, sep='\t', nrows=nrows)

    print(f"Original dataframe shape: {df.shape}")
    print(f"Columns: {list(df.columns)}")

    # Check if required columns exist
    if 'input' not in df.columns or 'target' not in df.columns:
        print("Error: 'input' and 'target' columns not found!")
        return None, None

    # Apply spell correction ONLY to input column
    print("Applying spell correction to 'input' column...")
    df['input_corrected'] = df['input'].astype(str).apply(
        lambda x: correct_text(x, hunspell_obj)
    )

    # Calculate accuracy metrics
    print("Calculating accuracy metrics...")
    metrics = calculate_metrics(df)

    return df, metrics


In [ ]:

def print_metrics(metrics):
    """Print formatted accuracy metrics"""
    print("\n" + "="*60)
    print("SPELL CORRECTION ACCURACY METRICS")
    print("="*60)
    print(f"Total samples: {metrics['total_samples']}")
    print(f"Exact match accuracy: {metrics['exact_match_accuracy']:.4f} ({metrics['exact_match_accuracy']*100:.2f}%)")
    print(f"Word-level accuracy: {metrics['word_level_accuracy']:.4f} ({metrics['word_level_accuracy']*100:.2f}%)")
    print(f"Character-level accuracy: {metrics['character_level_accuracy']:.4f} ({metrics['character_level_accuracy']*100:.2f}%)")
    print(f"Perfect corrections: {metrics['perfect_corrections']}/{metrics['total_samples']} ({metrics['perfect_correction_rate']:.2f}%)")
    print("="*60)


In [ ]:

def analyze_errors(df, show_top_n=10):
    """Analyze and display common error patterns"""

    # Get failed corrections
    failed_corrections = df[df['exact_match'] == 0].copy()

    if len(failed_corrections) == 0:
        print("\nNo errors found! Perfect performance!")
        return

    print(f"\n" + "="*60)
    print("ERROR ANALYSIS")
    print("="*60)
    print(f"Total errors: {len(failed_corrections)}")

    # Show worst performing samples
    print(f"\nTop {show_top_n} samples with lowest accuracy:")
    worst_samples = failed_corrections.nsmallest(show_top_n, 'char_accuracy')

    for idx, row in worst_samples.iterrows():
        print(f"\nOriginal: '{row['input']}'")
        print(f"Corrected: '{row['input_corrected']}'")
        print(f"Target: '{row['target']}'")
        print(f"Character accuracy: {row['char_accuracy']:.3f}")


In [ ]:

# Main execution
if __name__ == "__main__":
    # Initialize Hunspell
    spanish_hunspell = setup_spanish_hunspell()

    if spanish_hunspell:
        # Test the spell checker
        test_words = ["hola", "mundo", "kasa", "komputadora", "español"]
        print("\nTesting spell correction:")
        for word in test_words:
            corrected = correct_word(word, spanish_hunspell)
            print(f"{word} -> {corrected}")

        # Process your TSV file
        tsv_file_path = "train.tsv"  # UPDATE THIS PATH

        if os.path.exists(tsv_file_path):
            result_df, metrics = process_tsv_file(tsv_file_path, spanish_hunspell, nrows=10000)

            if result_df is not None and metrics is not None:
                # Print accuracy metrics
                print_metrics(metrics)

                # Save results
                output_path = "corrected_" + os.path.basename(tsv_file_path)
                result_df.to_csv(output_path, sep='\t', index=False)
                print(f"\nResults saved to: {output_path}")

                # Show sample results (corrected columns only)
                print("\nSample results:")
                display_cols = ['input', 'input_corrected', 'target', 'exact_match', 'word_accuracy', 'char_accuracy']
                available_cols = [col for col in display_cols if col in result_df.columns]
                print(result_df[available_cols].head(10))

                # Analyze errors
                analyze_errors(result_df, show_top_n=5)

        else:
            print(f"File {tsv_file_path} not found. Please update the file path.")
    else:
        print("Could not initialize Hunspell. Please check installation.")


Spanish Hunspell initialized successfully!

Testing spell correction:
hola -> hola
mundo -> mundo
kasa -> jasa
komputadora -> computadora
español -> español
Reading TSV file: train.tsv
Original dataframe shape: (10000, 7)
Columns: ['input', 'target', 'source_rule', 'freq', 'edit_count', 'seed', 'freq_bucket']
Applying spell correction to 'input' column...
Calculating accuracy metrics...

SPELL CORRECTION ACCURACY METRICS
Total samples: 10000
Exact match accuracy: 0.7260 (72.60%)
Word-level accuracy: 0.7260 (72.60%)
Character-level accuracy: 0.8589 (85.89%)
Perfect corrections: 7260.0/10000 (72.60%)

Results saved to: corrected_train.tsv

Sample results:
         input input_corrected       target  exact_match  word_accuracy  \
0     intranel        intranet     intranet          1.0            1.0   
1  evrasiático     eurasiático  eurasiático          1.0            1.0   
2    coñsumido       consumido    consumido          1.0            1.0   
3     posadaas         posadas      po

In [ ]:

# Alternative installation commands for different environments:
print("\n" + "="*50)
print("INSTALLATION COMMANDS FOR DIFFERENT ENVIRONMENTS:")
print("="*50)
print("""
FOR WINDOWS:
1. pip install pyhunspell
2. If above fails, try: pip install pyhunspell-windows
3. Or use conda: conda install hunspell

FOR KAGGLE:
!pip install pyhunspell

FOR COLAB:
!pip install pyhunspell
!apt-get install -y hunspell hunspell-es

FOR UBUNTU/LINUX:
sudo apt-get install hunspell libhunspell-dev
pip install pyhunspell
""")